In [ ]:
# Environment Sync & Auto-Dataset Check

import os, sys
from pathlib import Path
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

REPO_NAME = "food-classification-deep-learning"
BRANCH = "feature/nirmana-custom-cnn"

if 'google.colab' in sys.modules:
    print("[INFO] Running in Google Colab environment.")
    if not os.path.exists(f"/content/{REPO_NAME}"):
        !git clone -b {BRANCH} https://github.com/niRmana11/food-classification-deep-learning.git
        %cd /content/{REPO_NAME}
    else:
        %cd /content/{REPO_NAME}
        !git checkout {BRANCH}
        !git pull origin {BRANCH}
    
    if f"/content/{REPO_NAME}" not in sys.path:
        sys.path.insert(0, f"/content/{REPO_NAME}")

# Verify GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"[SUCCESS] GPU active: {gpus[0].name}")
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
else:
    print("[WARNING] No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU")

# Auto-download dataset if missing
if not os.path.exists("data/raw/food-101/images"):
    !python -m src.data.download_food101


In [ ]:
# Instantiate Standardized Data Loaders

from src.preprocessing.data_loader import get_food101_datasets
from src.models.custom_cnn import build_custom_cnn

BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)

print("[INFO] Loading datasets via shared factory...")
train_ds, val_ds, test_ds = get_food101_datasets(
    data_dir="data/raw/food-101",
    splits_dir="data/splits",
    model_type="custom_cnn",   # Automatically applies [0.0, 1.0] scaling
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)

print("[SUCCESS] Train, Validation, and Test pipelines loaded successfully.")


In [ ]:
# Model Instantiation & Checkpoints

RESULTS_DIR = Path("results/custom_cnn")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

model = build_custom_cnn(input_shape=(224, 224, 3), num_classes=101, learning_rate=0.001)

# Save architecture summary to model_summary.txt
with open(RESULTS_DIR / "model_summary.txt", "w") as f:
    model.summary(print_fn=lambda x: f.write(x + "\n"))

# Standardized Callbacks
callbacks = [
    # Stop if validation loss doesn't improve for 5 epochs
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    # Reduce learning rate when plateauing
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-5,
        verbose=1
    ),
    # Log epoch history to CSV
    tf.keras.callbacks.CSVLogger(
        filename=str(RESULTS_DIR / "history.csv"),
        separator=",",
        append=False
    )
]

print("[INFO] Callbacks configured. Ready for training.")
model.summary()


In [ ]:
# Execute Training on Colab GPU

EPOCHS = 20

print(f"[INFO] Starting Custom CNN training for up to {EPOCHS} epochs...")
start_time = time.time()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

total_training_time = time.time() - start_time
print(f"\n[SUCCESS] Training finished in {total_training_time:.1f} seconds ({total_training_time/60:.1f} minutes).")


In [ ]:
# Save Weights Locally & Permanently Backup to Google Drive

import os
import shutil
from google.colab import drive

# 1. Save weights locally in the results folder
WEIGHTS_LOCAL = "results/custom_cnn/custom_cnn_best.weights.h5"
model.save_weights(WEIGHTS_LOCAL)
print(f"[SUCCESS] Weights saved locally at: {WEIGHTS_LOCAL}")

# 2. Backup to Google Drive so they are never lost if Colab disconnects
print("[INFO] Mounting Google Drive for permanent backup...")
drive.mount('/content/drive')

GDRIVE_DIR = "/content/drive/MyDrive/food101_checkpoints"
os.makedirs(GDRIVE_DIR, exist_ok=True)
shutil.copy(WEIGHTS_LOCAL, GDRIVE_DIR)
print(f"[SUCCESS] Weights permanently backed up to: {GDRIVE_DIR}/custom_cnn_best.weights.h5")


In [ ]:
# Plot Training Curves (Figure for Report)

hist_df = pd.DataFrame(history.history)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss Plot
ax1.plot(hist_df["loss"], label="Training Loss", color="#2b5c8f", linewidth=2)
ax1.plot(hist_df["val_loss"], label="Validation Loss", color="#e28743", linewidth=2)
ax1.set_title("Custom CNN: Loss Progression", fontsize=12, fontweight='bold')
ax1.set_xlabel("Epoch", fontsize=10)
ax1.set_ylabel("Cross-Entropy Loss", fontsize=10)
ax1.legend()

# Accuracy Plot
ax1_acc = ax2.plot(hist_df["accuracy"], label="Training Accuracy", color="#2b5c8f", linewidth=2)
ax2.plot(hist_df["val_accuracy"], label="Validation Accuracy", color="#e28743", linewidth=2)
ax2.set_title("Custom CNN: Accuracy Progression", fontsize=12, fontweight='bold')
ax2.set_xlabel("Epoch", fontsize=10)
ax2.set_ylabel("Accuracy", fontsize=10)
ax2.legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / "training_curves.png", dpi=300)
plt.show()


# Final Test Set Evaluation & Export metrics.json
print("[INFO] Evaluating best restored weights on the UNSEEN test set...")
test_results = model.evaluate(test_ds)

# Profile inference latency over 500 images
print("[INFO] Measuring inference latency (ms/image)...")
latencies = []
for imgs, _ in test_ds.take(16): # ~512 images
    t0 = time.time()
    _ = model(imgs, training=False)
    latencies.append((time.time() - t0) / len(imgs) * 1000)
avg_latency = float(np.mean(latencies))

metrics_data = {
    "model_name": "Custom_CNN",
    "total_parameters": int(model.count_params()),
    "trainable_parameters": int(sum([tf.size(w).numpy() for w in model.trainable_weights])),
    "training_time_seconds": round(total_training_time, 2),
    "inference_latency_ms_per_image": round(avg_latency, 2),
    "final_val_loss": round(float(hist_df["val_loss"].min()), 4),
    "final_val_accuracy": round(float(hist_df["val_accuracy"].max()), 4),
    "final_test_loss": round(float(test_results[0]), 4),
    "final_test_accuracy": round(float(test_results[1]), 4)
}

with open(RESULTS_DIR / "metrics.json", "w") as f:
    json.dump(metrics_data, f, indent=2)


print("CUSTOM CNN FINAL EVALUATION SUMMARY")
for k, v in metrics_data.items():
    print(f"{k:<35}: {v}")

